In [57]:
import numpy as np

In [58]:
class MultinomialRegression:
      def __init__(self, lr=0.1, iterations=500):
        self.lr = lr
        self.iterations = iterations
        self.W = None
        self.b = None

      def softmax(self,z):
        exp_z= np.exp(z)
        return  exp_z/np.sum(exp_z, axis=1, keepdims=True)
      
      def one_hot(self, y, num_classes):
        
        y = np.array(y, dtype=int)
        identity = np.eye(num_classes)
    
        # Use y to PICK rows from the identity matrix
        one_hot = identity[y]
        return one_hot
        
      def compute_loss(Self,y_true,y_hat):
         epsilon = 1e-15  # small value to avoid log(0)
         y_hat = np.clip(y_hat, epsilon, 1 - epsilon)

        # Since y_true is one-hot, this picks only the correct class probability
         loss_per_sample = np.sum(y_true * np.log(y_hat), axis=1)
         loss = - np.mean(loss_per_sample)

         return loss
      
      def compute_gradients(self, X, y_true, y_hat):
  
        n_samples = X.shape[0]
        dz = y_hat - y_true  

        
        dW = (X.T @ dz) / n_samples  
        db = np.sum(dz, axis=0) / n_samples   

        return dW, db
      
      def fit(self, X, y):
   
        # Get dimensions from data
        n_samples, n_features = X.shape
        n_classes = 10

        # Initialize weights and bias (now we know shapes)
        self.W = np.zeros((n_features, n_classes))
        self.b = np.zeros(n_classes)

        # Convert labels to one-hot encoding
        y_one_hot = self.one_hot(y, n_classes)

        # Training loop
        for i in range(self.iterations):

            z = X @ self.W + self.b             
            y_hat = self.softmax(z)             

            #  Compute loss 
            if i % 100 == 0:
                loss = self.compute_loss(y_one_hot, y_hat)
                print(f"Iteration {i}, Loss: {loss:.4f}")

            #  Compute gradients
            dW, db = self.compute_gradients(X, y_one_hot, y_hat)

            #  Update parameters
            self.W -= self.lr * dW
            self.b -= self.lr * db

      def predict(self, X):
        if self.W is None:
            raise ValueError("Model is not trained yet. Call fit() first.")

        # Compute scores
        z = X @ self.W + self.b

        y_hat = self.softmax(z)

        # Pick class with highest probability
        y_pred = np.argmax(y_hat, axis=1)

        return y_pred





In [59]:
from preprocessing2 import preprocess
X_train, y_train, X_val, y_val, X_test, y_test, _ = preprocess( feature_method="hog" , n_pca=100)

model = MultinomialRegression(lr=0.5, iterations=1000)
model.fit(X_train, y_train)

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Iteration 0, Loss: 2.3026
Iteration 100, Loss: 0.6038
Iteration 200, Loss: 0.4391
Iteration 300, Loss: 0.3729
Iteration 400, Loss: 0.3350
Iteration 500, Loss: 0.3096
Iteration 600, Loss: 0.2911
Iteration 700, Loss: 0.2768
Iteration 800, Loss: 0.2652
Iteration 900, Loss: 0.2556


In [60]:
def evaluate_model(X, y, model, dataset_name="Validation"):
    """
    Evaluate multiclass model (digits 0-9)
    """

    y_pred = model.predict(X)
    y_true = y

    n_classes = 10  # MNIST

    # 🔹 Confusion Matrix
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1

    # 🔹 Accuracy
    accuracy = np.trace(cm) / np.sum(cm)

    # 🔹 Precision, Recall, F1 per class
    precision = np.zeros(n_classes)
    recall = np.zeros(n_classes)
    f1 = np.zeros(n_classes)

    for i in range(n_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp

        precision[i] = tp / (tp + fp + 1e-9)
        recall[i]    = tp / (tp + fn + 1e-9)
        f1[i]        = 2 * precision[i] * recall[i] / (precision[i] + recall[i] + 1e-9)

    # 🔹 Macro F1
    macro_f1 = np.mean(f1)

    # 🔹 Print results
    print(f"\n--- {dataset_name} Results ---")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Macro F1  : {macro_f1:.4f}")

    # 🔹 Confusion Matrix
    print("\nConfusion Matrix:")
    print(cm)

    # 🔹 Classification Report
    print("\nClassification Report:")
    print(f"{'Class':<10}{'Precision':>10}{'Recall':>10}{'F1-score':>12}")

    for i in range(n_classes):
        print(f"{i:<10}{precision[i]:>10.2f}{recall[i]:>10.2f}{f1[i]:>12.2f}")

In [61]:
evaluate_model(X_val, y_val, model, "Validation")
evaluate_model(X_test, y_test, model, "Test")


--- Validation Results ---
Accuracy  : 0.9513
Macro F1  : 0.9512

Confusion Matrix:
[[564   6   9   1   0   3   1   0   1   2]
 [  0 615   4   1   3   0   1   1   5   0]
 [  3   0 576   4   5   1   2   5   4   0]
 [  0   0  14 599   1   6   0   1   3   3]
 [  0   5   0   0 568   0   2   5   1  14]
 [  1   2   1  17   0 508   2   1  12   5]
 [  1   0   1   0   4   2 562   0   0   1]
 [  1   0   6   1   5   0   1 629   1  24]
 [  6   4   7   5   2   3   4   2 559   5]
 [  6   3   0   8   9   3   0  15   4 528]]

Classification Report:
Class      Precision    Recall    F1-score
0               0.97      0.96        0.96
1               0.97      0.98        0.97
2               0.93      0.96        0.95
3               0.94      0.96        0.95
4               0.95      0.95        0.95
5               0.97      0.93        0.95
6               0.98      0.98        0.98
7               0.95      0.94        0.95
8               0.95      0.94        0.94
9               0.91      0.92